In [1]:
import argparse
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Ellipse
from matplotlib.backends.backend_pdf import PdfPages

In [2]:
COUNTY_URL = (
    "https://www2.census.gov/geo/tiger/GENZ2024/shp/"
    "cb_2024_us_county_500k.zip"
)
SOLAR_MW_PER_KM2 = 35
CAPACITY_FACTOR = 0.20
HOURS_PER_YEAR = 8760

In [3]:
def load_nc_counties(solar_land_share):
    counties = gpd.read_file(COUNTY_URL)
    nc_counties = counties.loc[counties["STATEFP"] == "37"].to_crs(6543).copy()
    nc_counties = nc_counties.sort_values("NAME").reset_index(drop=True)

    # convert it to square kilometers.
    nc_counties["land_km2"] = nc_counties["ALAND"] / 1_000_000
    nc_counties["solar_km2"] = nc_counties["land_km2"] * solar_land_share
    nc_counties["solar_mw"] = nc_counties["solar_km2"] * SOLAR_MW_PER_KM2
    nc_counties["annual_mwh"] = (
        nc_counties["solar_mw"] * HOURS_PER_YEAR * CAPACITY_FACTOR
    )
    return nc_counties

In [5]:
def choose_counties(nc_counties, county):
    names = list(nc_counties["NAME"])
    if county.casefold() == "all":
        return names
    matching = [name for name in names if name.casefold() == county.casefold()]
    if not matching:
        raise ValueError(
            f"Unknown NC county: {county}. "
            "Use a North Carolina county name (without 'County') or 'all'."
        )
    return matching

In [7]:
def county_map(nc_counties, county, ax):
    selected = nc_counties[nc_counties["NAME"] == county]
    selected.plot(ax=ax, edgecolor="black", facecolor="#7b9b90")
    ax.set_axis_off()

def locator_map(nc_counties, county, ax):
    selected = nc_counties[nc_counties["NAME"] == county]
    nc_counties.plot(ax=ax, edgecolor="gray", facecolor="white")
    selected.plot(ax=ax, edgecolor="black", facecolor="#7b9b90")
    ax.set_axis_off()

In [8]:
def county_pie(nc_counties, county, ax):
    selected = nc_counties[nc_counties["NAME"] == county]
    land = selected["land_km2"].iloc[0]
    solar_land = selected["solar_km2"].iloc[0]
    other_land = land - solar_land

    ax.pie(
        [solar_land, other_land],
        labels=["Solar scenario", "Other land"],
        colors=["gold", "olivedrab"],
        autopct="%.1f%%",
        startangle=90,
    )
    ax.set_title("Hypothetical solar land use")

In [9]:
def county_info(nc_counties, county, solar_land_share, ax):
    selected = nc_counties[nc_counties["NAME"] == county]
    solar_area = selected["solar_km2"].iloc[0]
    solar_capacity = selected["solar_mw"].iloc[0]
    annual_energy = selected["annual_mwh"].iloc[0]

    mountain = Polygon(
        [(0.02, 0.60), (0.09, 0.80), (0.14, 0.69),
         (0.19, 0.76), (0.25, 0.60)],
        facecolor="olivedrab", transform=ax.transAxes,
    )
    ax.add_patch(mountain)
    sun = Ellipse(
        (0.13, 0.30), width=0.15, height=0.15,
        facecolor="gold", transform=ax.transAxes,
    )
    ax.add_patch(sun)

    ax.text(
        0.32, 0.80,
        f"SOLAR LAND SCENARIO\n{solar_area:,.2f} km²\n"
        f"({solar_land_share:.0%} of county land)",
        transform=ax.transAxes, fontsize=12, color="olivedrab", va="top",
    )
    ax.text(
        0.32, 0.40,
        f"ESTIMATED CAPACITY\n{solar_capacity:,.0f} MW\n"
        f"ANNUAL ENERGY\n{annual_energy:,.0f} MWh",
        transform=ax.transAxes, fontsize=12, color="darkgoldenrod", va="top",
    )
    ax.set_axis_off()

In [10]:
def county_figure(nc_counties, county, solar_land_share, pdf):
    """Make one atlas page for a county."""
    fig = plt.figure(figsize=(12, 8))
    ax_county = fig.add_axes([0.05, 0.23, 0.55, 0.54])
    ax_locator = fig.add_axes([0.05, 0.08, 0.25, 0.12])
    ax_pie = fig.add_axes([0.65, 0.48, 0.30, 0.30])
    ax_info = fig.add_axes([0.65, 0.12, 0.30, 0.32])

    county_map(nc_counties, county, ax_county)
    locator_map(nc_counties, county, ax_locator)
    county_pie(nc_counties, county, ax_pie)
    county_info(nc_counties, county, solar_land_share, ax_info)

    fig.text(0.05, 0.94, "NORTH CAROLINA / County Solar Energy Potential Atlas",
             fontsize=10, color="olivedrab")
    fig.text(0.05, 0.87, f"{county} County", fontsize=28, fontweight="bold")
    fig.text(
        0.05, 0.03,
        "Source: U.S. Census Bureau (ALAND) | Scenario, not measured solar potential",
        fontsize=8, color="gray",
    )
    fig.text(
        0.05, 0.005,
        f"Assumptions: {solar_land_share:.0%} of land; "
        f"{SOLAR_MW_PER_KM2:g} MW/km²; {CAPACITY_FACTOR:.0%} capacity factor",
        fontsize=8, color="gray",
    )
    pdf.savefig(fig)
    plt.close(fig)

In [11]:
def generate_atlas(county, solar_land_share=0.01, output_dir="."):
    """Produce a county PDF atlas and estimates CSV."""
    if not 0 <= solar_land_share <= 1:
        raise ValueError("--solar-land-share must be between 0 and 1.")

    nc_counties = load_nc_counties(solar_land_share)
    selected_counties = choose_counties(nc_counties, county)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    pdf_path = output_dir / "NC_County_Solar_Atlas.pdf"
    csv_path = output_dir / "NC_County_Solar_Estimates.csv"

    with PdfPages(pdf_path) as pdf:
        for selected_county in selected_counties:
            county_figure(nc_counties, selected_county, solar_land_share, pdf)

    nc_counties.loc[
        nc_counties["NAME"].isin(selected_counties),
        ["NAME", "land_km2", "solar_km2", "solar_mw", "annual_mwh"],
    ].to_csv(csv_path, index=False)

    print(f"Saved {len(selected_counties)} atlas page(s) to {pdf_path}")
    print(f"Saved solar scenario estimates to {csv_path}")

In [12]:
def main():
    parser = argparse.ArgumentParser(
        description="Create a North Carolina county solar energy potential atlas."
    )
    parser.add_argument(
        "county", help="Required NC county name (e.g., Alamance) or 'all'"
    )
    parser.add_argument(
        "--solar-land-share", type=float, default=0.01,
        help="Optional fraction of county land for hypothetical solar use (default: 0.01 = 1%%)",
    )
    args = parser.parse_args()
    try:
        generate_atlas(args.county, args.solar_land_share)
    except (ValueError, OSError) as exc:
        parser.error(str(exc))

In [14]:
county = "Alamance"  
solar_land_share = 0.01 
generate_atlas(county, solar_land_share)

Saved 1 atlas page(s) to NC_County_Solar_Atlas.pdf
Saved solar scenario estimates to NC_County_Solar_Estimates.csv


I used AI most of this assignment. I was not in class to learn about refactoring, generalizing, and coverting, and am confused on what to do. However, I learned further how to create a .py as I faced some issues turning my jupyter notebook into a .py and had to go step by step to turn this into a .py. 
